This notebook allows you to run ACOLITE processing on Google Colab. The required dependencies are installed and the latest version is cloned from GitHub (https://github.com/acolite/acolite).

In this example, a PlanetScope image is retrieved from ESA Sample Data (https://earth.esa.int/eogateway/missions/planetscope/sample-data), processed and stored on your Google Drive. Ancillary data can be retrieved if EarthData credentials are set up, but are not essential for this example.

Author: Quinten Vanhellemont, RBINS, quinten.vanhellemont@naturalsciences.be

The next cell sets up the ACOLITE dependencies in Colab if they are not already there. This is specifical to Colab and you will need to set up an ACOLITE environment if you would want to run it locally.

In [ ]:
# general imports
import os, sys

# install dependencies for acolite
# for colab: assume if netCDF4 is installed the other dependencies are as well
try:
  import netCDF4
except:
  !pip install numpy matplotlib scipy gdal pyproj scikit-image pyhdf pyresample netcdf4 h5py requests pygrib cartopy

In [ ]:
# clone acolite repo if it does not exist
if not os.path.exists('acolite'):
  !git clone --depth 1 https://github.com/acolite/acolite

In [ ]:
# add acolite to path and import
sys.path.append('acolite')
import acolite as ac

The next cell downloads a sample SuperDove scene kindly provided by ESA.

In [ ]:
## download sample SuperDove scene from ESA
scene_url = 'https://earth.esa.int/eogateway/ftp/missions/sample-data/third-party-missions/planetscope/PSScene_Basic_Analytic_8b_udm2.zip'

# make input directory
if not os.path.exists('Input/'): os.makedirs('Input/')
local_scene_zip = '{}/{}'.format('Input', os.path.basename(scene_url))
local_scene = '{}/{}'.format('Input', os.path.basename(scene_url)[0:-4])

## download scene
if not os.path.exists(local_scene_zip):
  !wget -P 'Input' $scene_url

## unzip file
if not os.path.exists(local_scene):
  !unzip -d 'Input' $local_scene_zip

## find file
import glob
inputfile = glob.glob('{}/**/basic_analytic_8b_udm2'.format(local_scene))[0]


In [ ]:
## location of interest
## here for centering 3x3 km box and extracting spectra
region_name = 'EtangDeLOlivier'
station_lon = 4.9905
station_lat = 43.5257

The next cell sets up a Python dictionary with typical settings for processing ACOLITE/DSF for a region of interest (here a square box centred on a location).

In [ ]:
# acolite settings
settings = {## basic input and output settings
            ## local_scenes list is used as returned from the API download function
            'inputfile': inputfile,
            'output': 'Output/',
            'l2w_parameters': ['Rrs_*'],
            'l2w_rgb_keys': ['Rrs'],
            'rgb_Rrs': True,

            ## set up region of interest centred on lon/lat, with box size in km
            'region_name': region_name,
            'station_lon': station_lon,
            'station_lat': station_lat,
            'station_box_size': 3,
            'station_box_units': 'km',

            ## disable ancillary data retrieval, set to True if EarthData credentials are set
            'ancillary_data': False,

            ## use DSF processing
            'atmospheric_correction_method': 'dark_spectrum',

            ## disable negatives masking
            'l2w_mask_negative_rhow': False,

            ## delete L1R output NetCDF
            'l1r_delete_netcdf': True,
            ## don't output L1R RGBs
            'l1r_rgb_keys': [],
            ## delete ACOLITE text outputs
            'delete_acolite_run_text_files': True,
            }

The next cell passes the settings dictionary to ACOLITE and runs the processing. Note that the first run will be slightly slower, as the LUTs for the current sensor will have to be downloaded.

In [ ]:
## run acolite with settings dict
ret = ac.acolite.acolite_run(settings)

In [ ]:
## imports for plotting
import glob
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
## create rhos RGB
ncf = glob.glob('Output/*L2R*.nc')[0]

gatts = ac.shared.nc_gatts(ncf)
title = '{} {}'.format(gatts['sensor'].replace('_', '/'), gatts['isodate'][0:19])

lat = ac.shared.nc_data(ncf, 'lat')
lon = ac.shared.nc_data(ncf, 'lon')
datasets = ac.shared.nc_datasets(ncf)

stack = []
waves = []
for ds in datasets:
  if ds.startswith('rhos_'):
      waves.append(float(ds[ds.find('_')+1:]))
      stack.append(ac.shared.nc_data(ncf, ds))
stack = np.asarray(stack)
waves = np.asarray(waves)
rgb = ac.map.rgb.stack(waves, stack,  rgb_max = [0.15] * 3,)

fig, ax = plt.subplots()
plt.xlabel('Longitude (°E)')
plt.ylabel('Latitude (°N)')
plt.title(title)

mesh = plt.pcolormesh(lon, lat, rgb)
plt.plot(station_lon, station_lat, 'x', color = 'Red')
plt.savefig('Output/RGB.png', dpi = 300, bbox_inches = 'tight', facecolor = 'white')

In [ ]:
## extract Rrs spectrum
ncf = glob.glob('Output/*L2W*.nc')[0]
ext = ac.shared.nc_extract_point(ncf, station_lon, station_lat, box_size = 11)

## extract rhot and rhos spectra
ncf = glob.glob('Output/*L2R*.nc')[0]
ext2 = ac.shared.nc_extract_point(ncf, station_lon, station_lat, box_size = 11)

In [ ]:
## plot Rrs
title = '{} {}'.format(ext['gatts']['sensor'].replace('_', '/'), ext['gatts']['isodate'][0:19])
mean = np.asarray([np.nanmean(ext['data'][ds]) for ds in ext['Rrs_datasets']])
std = np.asarray([np.nanstd(ext['data'][ds]) for ds in ext['Rrs_datasets']])

plt.errorbar(ext['Rrs_wave'], mean, marker = '.', linestyle = '--', yerr = std, label = 'Rrs')
plt.xlabel('Wavelength (nm)')
plt.ylabel(r'Rrs ($sr^{-1}$)')
plt.ylim(-0.002, 0.061)

plt.legend()

plt.title(title)

plt.savefig('Output/Rrs.png', dpi = 300, bbox_inches = 'tight', facecolor = 'white')

In [ ]:
## plot Rrs and rhot (/pi)
title = '{} {}'.format(ext['gatts']['sensor'].replace('_', '/'), ext['gatts']['isodate'][0:19])
meant = np.asarray([np.nanmean(ext2['data'][ds]) for ds in ext2['rhot_datasets']])
stdt = np.asarray([np.nanstd(ext2['data'][ds]) for ds in ext2['rhot_datasets']])

plt.errorbar(ext['Rrs_wave'], mean, marker = '.', linestyle = '--', yerr = std, label = 'Rrs')
plt.errorbar(ext2['rhot_wave'], meant/np.pi, marker = '.', linestyle = '--', yerr = std/np.pi, label = 'TOA')

plt.xlabel('Wavelength (nm)')
plt.ylabel(r'Rrs ($sr^{-1}$)')
plt.ylim(-0.002, 0.061)

plt.legend()
plt.title(title)

plt.savefig('Output/Rrs_TOA.png', dpi = 300, bbox_inches = 'tight', facecolor = 'white')